# Intro to Dash - BI Forum '25

## Imports

In [ ]:
# importing the needed libraries
from dash import Dash # to initialize our app
from dash import html, dcc # to create the layout and add interaction
from dash import callback, Output, Input # to define how the interaction will work
import plotly.express as px # to create our charts
import pandas as pd # to get data
import requests # for getting the geojson for later

## Dataset

In [ ]:
# reading the world happiness dataset from git
df = pd.read_csv('https://raw.githubusercontent.com/CzibiBIC/biforum25/refs/heads/main/datasets/whr_with_iso3.csv')

df.head()

## Example chart

We want to create a top 5 bar chart that visualizes the happiest countries of a selected year based on their life ladder score

In [ ]:
# firstly let's create a rank column
df['Yearly Ranking'] = df.groupby('year')['Life Ladder'].rank(method='first',ascending=False)

In [ ]:
# lets check the result
df.query('`Country name`== "Hungary"').head()

In [ ]:
df2 = df.query('year == 2007 and `Yearly Ranking` < 6').sort_values('Yearly Ranking', ascending = False)

df2

In [ ]:
top_5_bar = px.bar(
        df2, x = 'Life Ladder', y = 'Country name',
        title = 'Top 5 happiest countries',
        hover_name = 'Country name',
        template = 'simple_white',
        color = 'Healthy life expectancy at birth',
        width = 900, height = 500
    )

top_5_bar.update_layout(title_x = 0.5, plot_bgcolor = 'white', yaxis_title = None)
top_5_bar.update_xaxes(range = [0,8])

top_5_bar.show()

## Creating our first Dash app

### Defining the app

Available core components:
https://dash.plotly.com/dash-core-components

Available html components:
https://dash.plotly.com/dash-html-components

In [ ]:
## Creating an app object
app = Dash(__name__)

In [ ]:
# defining a layout
app.layout = html.Div(children=[
    html.H1(children='World Happiness Dashboard', style={'textAlign':'center'}),

    html.Div(
     children = [
        dcc.Graph(
          id = 'top_5_happiest_countries',
          figure = top_5_bar
      )
    ])
], style={'backgroundColor': 'white'})

app.run()

### Adding a dropdown to the app

In [ ]:
app = Dash(__name__)

app.layout = html.Div(children=[
    html.H1(children='World Happiness Dashboard', style={'textAlign':'center'}),
    
    html.Div(
     children = [
         html.Label(
         'Select Year:',
          style={'fontWeight': 'bold', 'marginBottom': '5px', 'display': 'block'}
         ),
         dcc.Dropdown(
          id = 'year_dropdown',
          options= sorted(set(df['year'])),
          value=2005,
          style = {'width': '35%',
                  'borderWidth': '3px', 'borderRadius': '5px',
                  'borderColor' : 'black'}
        )
    ]),
    
    html.Div(
     children = [
        dcc.Graph(
          id = 'top_5_happiest_countries',
          figure = top_5_bar
      )
    ])
], style={'backgroundColor': 'white'})

app.run()

### Adding interactivity

In [ ]:
app = Dash(__name__)

app.layout = html.Div(children=[
    html.H1(children='World Happiness Dashboard', style={'textAlign':'center'}),
    
    html.Div(
     children = [
         html.Label(
         'Select Year:',
          style={'fontWeight': 'bold', 'marginBottom': '5px', 'display': 'block'}
         ),
         dcc.Dropdown(
          id = 'year_dropdown',
          options= sorted(set(df['year'])),
          value=2005,
          style = {'width': '35%',
                  'borderWidth': '3px', 'borderRadius': '5px',
                  'borderColor' : 'black'}
        )
    ]),
    
    html.Div(
     children = [
        dcc.Graph(
          id = 'top_5_happiest_countries',
          figure = top_5_bar
      )
    ])
], style={'backgroundColor': 'white'})

@app.callback(
    Output('top_5_happiest_countries', 'figure'),
    Input('year_dropdown', 'value')
)
def update_charts(selected_year):
    df_year = df.query('year == @selected_year and `Yearly Ranking` < 6').sort_values('Yearly Ranking', ascending = False)
    top_5_bar = px.bar(
        df_year, x = 'Life Ladder', y = 'Country name',
        title = 'Top 5 happiest countries',
        hover_name = 'Country name',
        template = 'simple_white',
        color = 'Healthy life expectancy at birth',
        width = 900, height = 500
    )

    top_5_bar.update_layout(title_x = 0.5, plot_bgcolor = 'white', yaxis_title = None)
    top_5_bar.update_xaxes(range = [0,8])

    return top_5_bar
    
app.run()

### Adding a second chart to the app

In [ ]:
# geojson for the map visualization
geojson_url = "https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson"

# Fetch the file
response = requests.get(geojson_url)
geojson_data = response.json()

In [ ]:
mapbox = px.choropleth_mapbox(
        df.query('year == 2007'),
        geojson = geojson_data,
        featureidkey='properties.ISO3166-1-Alpha-3',
        locations = 'ISO3',
        color='Life Ladder',
        color_continuous_scale='Viridis',
        hover_name = 'Country name',
        labels={'Life Ladder': 'Happiness Score'},
        mapbox_style = 'open-street-map',
        width = 1200
    )
    
mapbox.update_layout(
    mapbox=dict(
        center={"lat": 20, "lon": 0},  # roughly centers the whole world
        zoom=0.8                       # lower = more zoomed out
    ),
    margin={"r":0,"t":0,"l":0,"b":0}
)

In [ ]:
app = Dash(__name__)

app.layout = html.Div(children=[
    html.H1(children='World Happiness Dashboard', style={'textAlign':'center'}),
    
    html.Div(
     children = [
         html.Label(
         'Select Year:',
          style={'fontWeight': 'bold', 'marginBottom': '5px', 'display': 'block'}
         ),
         dcc.Dropdown(
          id = 'year_dropdown',
          options= sorted(set(df['year'])),
          value=2005,
          style = {'width': '35%',
                  'borderWidth': '3px', 'borderRadius': '5px',
                  'borderColor' : 'black'}
        )
    ]),
    
    html.Div(
     children = [
        dcc.Graph(
          id = 'top_5_happiest_countries',
          figure = top_5_bar
      ), # adding the second chart here
         dcc.Graph(
          id = 'mapbox',
          figure = mapbox
      )
    ], style={
        'display': 'flex',
        'flex-direction': 'row',
        'justify-content': 'center',
        'align-items': 'center',
    })
], style={'backgroundColor': 'white'})

@app.callback(
    # adding interactivity for the second chart as well
    [Output('top_5_happiest_countries', 'figure'),Output('mapbox', 'figure')],
    Input('year_dropdown', 'value')
)
def update_charts(selected_year):
    df_year = df.query('year == @selected_year and `Yearly Ranking` < 6').sort_values('Yearly Ranking', ascending = False)
    top_5_bar = px.bar(
        df_year, x = 'Life Ladder', y = 'Country name',
        title = 'Top 5 happiest countries',
        hover_name = 'Country name',
        template = 'simple_white',
        color = 'Healthy life expectancy at birth',
        width = 900, height = 500
    )

    top_5_bar.update_layout(title_x = 0.5, plot_bgcolor = 'white', yaxis_title = None)
    top_5_bar.update_xaxes(range = [0,8])

    mapbox = px.choropleth_mapbox(
        df.query('year == @selected_year'),
        geojson = geojson_data,
        featureidkey='properties.ISO3166-1-Alpha-3',
        locations = 'ISO3',
        color='Life Ladder',
        color_continuous_scale='Viridis',
        hover_name = 'Country name',
        labels={'Life Ladder': 'Happiness Score'},
        mapbox_style = 'open-street-map',
        width = 1200
    )
    
    mapbox.update_layout(
        mapbox=dict(
            center={"lat": 20, "lon": 0},  # roughly centers the whole world
            zoom=0.8                       # lower = more zoomed out
        ),
        margin={"r":0,"t":0,"l":0,"b":0}
    )
    
    return top_5_bar, mapbox
    
app.run()